# Query seeds.csv via BigQuery REST API

Uses requests with google.auth credentials.

In [ ]:

from pathlib import Path
import json
import pandas as pd
import requests
import google.auth
from google.auth.transport.requests import Request

base = Path('..').resolve()
seeds = pd.read_csv(base / 'data' / 'input' / 'seeds.csv')
txids = seeds['txid'].dropna().tolist()

credentials, project_id = google.auth.default()
credentials.refresh(Request())
headers = {'Authorization': f'Bearer {credentials.token}'}

sql = {
    'query': """
        SELECT TO_HEX(`hash`) AS txid, block_timestamp
        FROM `bigquery-public-data.crypto_bitcoin.transactions`
        WHERE TO_HEX(`hash`) IN UNNEST(@txid_list)
    """,
    'useLegacySql': False,
    'queryParameters': [
        {
            'name': 'txid_list',
            'parameterType': {'type': 'ARRAY', 'arrayType': {'type': 'STRING'}},
            'parameterValue': {'arrayValues': [{'value': t} for t in txids]}
        }
    ]
}

url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{project_id}/queries'
resp = requests.post(url, headers=headers, json=sql)
rows = resp.json().get('rows', [])
data = [(r['f'][0]['v'], r['f'][1]['v']) for r in rows]
pd.DataFrame(data, columns=['txid', 'block_timestamp'])
